In [52]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium
import glob
import shutil
from pathlib import Path

Import statsmodels as sm



SyntaxError: invalid syntax (3433885546.py, line 12)

In [27]:
citibike_df = pd.read_csv("../data/JC/JC2025_Enriched.csv")
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour,season
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,13.307467,2025-11-18,2025-11,November,Tuesday,18,Autumn
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,14.495367,2025-11-26,2025-11,November,Wednesday,16,Autumn
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member,6.983450,2025-11-04,2025-11,November,Tuesday,22,Autumn
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,5.803383,2025-11-08,2025-11,November,Saturday,6,Autumn
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,6.658383,2025-11-24,2025-11,November,Monday,20,Autumn


In [28]:
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

monthly_rides

,month,number_of_rides
0,2025-01,1
1,2025-02,45250
2,2025-03,73277
3,2025-04,81533
4,2025-05,93202
5,2025-06,96736
6,2025-07,107374
7,2025-08,108001
8,2025-09,115580
9,2025-10,103586


In [29]:
#number of rides per month

fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show() 

In [30]:
#number of rides per season

season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,93113
1,Spring,248012
2,Summer,312111
0,Autumn,295399


In [31]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

In [32]:
# Top 10 start stations

top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,42388
58,Hoboken Terminal - Hudson St & Hudson Pl,24723
93,River St & Newark St,21383
53,Hamilton Park,21139
84,Newport PATH,19554
18,Bergen Ave & Sip Ave,19116
44,Exchange Pl,19019
0,11 St & Washington St,18593
92,River St & 1 St,18031
2,14 St Ferry - 14 St & Shipyard Ln,17881


In [33]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

In [34]:
# Top 10 end stations 

top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(
        number_of_arrivals=("ride_id", "count")
    )
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
229,Grove St PATH,44918
238,Hoboken Terminal - Hudson St & Hudson Pl,25505
340,River St & Newark St,22113
230,Hamilton Park,21230
311,Newport PATH,19592
204,Exchange Pl,19119
71,Bergen Ave & Sip Ave,19079
7,11 St & Washington St,18610
312,Newport Pkwy,17846
10,14 St Ferry - 14 St & Shipyard Ln,17619


In [35]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

In [36]:
# Merge with weather data to see paterns

daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2025-01-31,1
1,2025-02-01,1437
2,2025-02-02,1101
3,2025-02-03,1795
4,2025-02-04,2060


In [42]:
weather_daily = pd.read_csv("../data/JC/jersey_weather_2025.csv")

In [43]:
weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [46]:
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.dtypes

temperature_2m_max            float64
temperature_2m_min            float64
temperature_2m_mean           float64
precipitation_sum             float64
rain_sum                      float64
snowfall_sum                  float64
wind_speed_10m_max            float64
date                   datetime64[us]
dtype: object

In [47]:
# Merging two datasets 

bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-31,1,6.2,0.7,3.6,10.3,10.3,0.00,13.5
1,2025-02-01,1437,6.2,-5.8,0.7,2.4,2.4,0.00,19.4
2,2025-02-02,1101,1.4,-9.6,-4.5,1.2,0.0,0.84,11.7
3,2025-02-03,1795,6.1,-3.1,1.9,0.3,0.0,0.21,12.8
4,2025-02-04,2060,6.2,-0.5,3.6,0.3,0.3,0.00,23.0


In [48]:
# Checking missing values after the merge

bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
rain_sum               0
snowfall_sum           0
wind_speed_10m_max     0
dtype: int64

In [49]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

ModuleNotFoundError: No module named 'statsmodels'